# Ecommerce Customer Segmentation
## Chapter 2 — Data Understanding & Preprocessing

This notebook implements the data-understanding and preprocessing stage of the project on the supplied **Online Retail.xlsx** dataset.

### Main tasks
1. Data loading and structure inspection
2. Missing-value analysis
3. Duplicate detection
4. Cancellation / return analysis
5. Invalid-value analysis
6. Transaction cleaning
7. Feature engineering (`Amount`)
8. Date feature extraction
9. Descriptive statistics and EDA
10. Customer-level aggregation
11. RFM construction

### Cleaning policy
For the main positive-sales customer-segmentation population:
- Remove exact duplicate rows.
- Keep records with a known `CustomerID`.
- Exclude invoices beginning with `C` (cancellations).
- Exclude `Quantity <= 0`.
- Exclude `UnitPrice <= 0`.
- Exclude missing `Description`.

Returns/cancellations are analyzed first and are not silently discarded without documentation.


In [ ]:
# 0. Setup
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_PATH = "Online Retail.xlsx"
OUTPUT_DIR = "outputs"
FIG_DIR = os.path.join(OUTPUT_DIR, "figures")
TABLE_DIR = os.path.join(OUTPUT_DIR, "tables")

for folder in [OUTPUT_DIR, FIG_DIR, TABLE_DIR]:
    os.makedirs(folder, exist_ok=True)

print("Ready.")


In [ ]:
# 1. Load the dataset
df = pd.read_excel(DATA_PATH, sheet_name="Online Retail")

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
display(df.head())


In [ ]:
# 2. Data types and missing values
print("Data types:")
display(df.dtypes.to_frame("dtype"))

missing = df.isna().sum().to_frame("missing_count")
missing["missing_pct"] = (missing["missing_count"] / len(df) * 100).round(2)
display(missing)

print("Exact duplicate rows:", df.duplicated().sum())


In [ ]:
# 3. Data-quality audit
df["IsCancelled"] = df["InvoiceNo"].astype(str).str.startswith("C")
df["Amount"] = df["Quantity"] * df["UnitPrice"]

audit = pd.DataFrame({
    "Metric": [
        "Rows", "Columns", "Unique invoices", "Unique stock codes",
        "Unique customers", "Unique countries",
        "Missing CustomerID", "Missing Description",
        "Exact duplicate rows", "Cancelled invoice rows",
        "Quantity <= 0", "UnitPrice <= 0", "Negative Amount"
    ],
    "Value": [
        len(df), df.shape[1], df["InvoiceNo"].nunique(),
        df["StockCode"].nunique(), df["CustomerID"].nunique(),
        df["Country"].nunique(), df["CustomerID"].isna().sum(),
        df["Description"].isna().sum(), df.duplicated().sum(),
        df["IsCancelled"].sum(), (df["Quantity"] <= 0).sum(),
        (df["UnitPrice"] <= 0).sum(), (df["Amount"] < 0).sum()
    ]
})

display(audit)
audit.to_csv(os.path.join(TABLE_DIR, "data_quality_audit.csv"), index=False)


In [ ]:
# 4. Cancellation / return analysis
cancelled = df["IsCancelled"]
negative_qty = df["Quantity"] < 0

return_analysis = pd.Series({
    "Cancelled rows": int(cancelled.sum()),
    "Negative quantity rows": int(negative_qty.sum()),
    "Cancelled + negative quantity": int((cancelled & negative_qty).sum()),
    "Negative quantity not marked C": int((~cancelled & negative_qty).sum()),
    "UnitPrice <= 0": int((df["UnitPrice"] <= 0).sum())
}, name="count")

display(return_analysis.to_frame())


In [ ]:
# 5. Cleaning
clean = df.copy()

rows_before = len(clean)

# Remove exact duplicates
clean = clean.drop_duplicates()

# Keep identifiable customers
clean = clean[clean["CustomerID"].notna()]

# Positive-sales population
clean = clean[~clean["IsCancelled"]]
clean = clean[clean["Quantity"] > 0]
clean = clean[clean["UnitPrice"] > 0]
clean = clean[clean["Description"].notna()]

clean["CustomerID"] = clean["CustomerID"].astype(int)
clean["Amount"] = clean["Quantity"] * clean["UnitPrice"]

print("Rows before cleaning:", rows_before)
print("Rows after cleaning:", len(clean))
print("Rows removed:", rows_before - len(clean))
print("Retention:", round(100 * len(clean) / rows_before, 2), "%")
print("Customers after cleaning:", clean["CustomerID"].nunique())


In [ ]:
# 6. Date features
clean["InvoiceDate"] = pd.to_datetime(clean["InvoiceDate"])

clean["Date"] = clean["InvoiceDate"].dt.date
clean["Time"] = clean["InvoiceDate"].dt.time
clean["Year"] = clean["InvoiceDate"].dt.year
clean["Month"] = clean["InvoiceDate"].dt.month
clean["Day"] = clean["InvoiceDate"].dt.day
clean["Hour"] = clean["InvoiceDate"].dt.hour

display(clean[[
    "InvoiceNo", "InvoiceDate", "Date", "Time",
    "Year", "Month", "Day", "Hour"
]].head())


In [ ]:
# 7. Descriptive statistics
display(clean[["Quantity", "UnitPrice", "Amount"]].describe())

print("Date range:")
print(clean["InvoiceDate"].min(), "to", clean["InvoiceDate"].max())


In [ ]:
# 8. EDA — Monthly revenue
monthly_sales = (
    clean.assign(MonthPeriod=clean["InvoiceDate"].dt.to_period("M"))
         .groupby("MonthPeriod")["Amount"]
         .sum()
)

display(monthly_sales.to_frame("Revenue"))

plt.figure(figsize=(11, 4.5))
monthly_sales.plot(marker="o")
plt.title("Monthly Revenue After Cleaning")
plt.xlabel("Month")
plt.ylabel("Revenue")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

monthly_sales.to_csv(os.path.join(TABLE_DIR, "monthly_revenue.csv"))


In [ ]:
# 9. EDA — Country revenue
country_summary = (
    clean.groupby("Country")
         .agg(
             Revenue=("Amount", "sum"),
             Orders=("InvoiceNo", "nunique"),
             Customers=("CustomerID", "nunique"),
             Rows=("InvoiceNo", "size")
         )
         .sort_values("Revenue", ascending=False)
)

display(country_summary.head(15))

top10 = country_summary.head(10).sort_values("Revenue")
plt.figure(figsize=(9, 5))
plt.barh(top10.index, top10["Revenue"])
plt.title("Top 10 Countries by Revenue")
plt.xlabel("Revenue")
plt.tight_layout()
plt.show()

country_summary.to_csv(os.path.join(TABLE_DIR, "country_summary.csv"))


In [ ]:
# 10. EDA — Transaction amount distribution
plt.figure(figsize=(8, 4.5))
plt.hist(clean["Amount"], bins=80)
plt.title("Transaction Amount Distribution")
plt.xlabel("Amount")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()


In [ ]:
# 11. Customer-level aggregation
customer_summary = (
    clean.groupby("CustomerID")
         .agg(
             Orders=("InvoiceNo", "nunique"),
             TotalQuantity=("Quantity", "sum"),
             TotalRevenue=("Amount", "sum"),
             AvgLineAmount=("Amount", "mean"),
             FirstPurchase=("InvoiceDate", "min"),
             LastPurchase=("InvoiceDate", "max")
         )
         .reset_index()
)

display(customer_summary.head())
display(customer_summary.describe(include="all"))
customer_summary.to_csv(os.path.join(TABLE_DIR, "customer_summary.csv"), index=False)


In [ ]:
# 12. RFM construction
reference_date = clean["InvoiceDate"].max() + pd.Timedelta(days=1)

rfm = (
    clean.groupby("CustomerID")
         .agg(
             Recency=("InvoiceDate", lambda x: (reference_date - x.max()).days),
             Frequency=("InvoiceNo", "nunique"),
             Monetary=("Amount", "sum"),
             First_Purchase=("InvoiceDate", "min"),
             Minimum=("Amount", "min"),
             Maximum=("Amount", "max"),
             Mean=("Amount", "mean")
         )
         .reset_index()
)

display(rfm.head())
display(rfm[["Recency", "Frequency", "Monetary"]].describe())

rfm.to_csv(os.path.join(TABLE_DIR, "rfm.csv"), index=False)


In [ ]:
# 13. RFM distributions
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

for ax, col in zip(axes, ["Recency", "Frequency", "Monetary"]):
    ax.boxplot(rfm[col])
    ax.set_title(col)
    ax.set_ylabel(col)

plt.suptitle("RFM Feature Distributions")
plt.tight_layout()
plt.show()


In [ ]:
# 14. IQR screening — descriptive only
def iqr_outliers(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    mask = (series < lower) | (series > upper)
    return {
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "Lower": lower,
        "Upper": upper,
        "OutlierCount": int(mask.sum())
    }

iqr_report = pd.DataFrame({
    col: iqr_outliers(rfm[col]) for col in ["Recency", "Frequency", "Monetary"]
}).T

display(iqr_report)

# Outlier removal is intentionally NOT done here.
# Chapter 11 will use dedicated outlier-detection methods.


## Chapter 2 Results

For the supplied dataset, the completed preprocessing produced:

| Measure | Result |
|---|---:|
| Raw rows | 541,909 |
| Exact duplicate rows removed | 5,268 |
| Missing CustomerID | 135,080 |
| Missing Description | 1,454 |
| Cancelled invoice rows | 9,288 |
| Quantity <= 0 | 10,624 |
| UnitPrice <= 0 | 2,517 |
| Clean positive-sales rows | 392,692 |
| Clean customers | 4,338 |
| Clean unique invoices | 18,532 |
| Clean revenue | 8,887,208.89 |

### Important methodological note
Returns/cancellations were analyzed before removal. They are excluded from the **main positive-sales RFM population** so that Frequency and Monetary describe purchasing behavior rather than net returns.

The resulting `rfm.csv` is the main customer-level input for the next segmentation phase.
